
# Test file for MDanalysis

In [58]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import MDAnalysis as mda
from MDAnalysis.analysis.distances import distance_array
from MDAnalysis.analysis import helix_analysis as hel

In [14]:
pdb_path = "../data/ompp-pdb/ompp_1.pdb"

## Plot confidence scores

In [ ]:
# Plot pLDDT and Cα–Cα distance map from a PDB file
u = mda.Universe(pdb_path)
ca = u.select_atoms("protein and name CA")
if len(ca) == 0:
    raise ValueError("No CA atoms found. Is this a protein PDB?")

plddt = ca.tempfactors  # AlphaFold pLDDT in B-factor field
coords = ca.positions
dmat = distance_array(coords, coords)

plt.figure()
plt.plot(np.arange(1, len(plddt) + 1), plddt)
plt.xlabel("Residue index (CA order)")
plt.ylabel("pLDDT (B-factor)")
plt.title("AlphaFold confidence along sequence")
plt.show()

plt.figure()
plt.imshow(dmat, aspect="auto")
plt.colorbar(label="Distance (Å)")
plt.title("Cα–Cα distance map")
plt.xlabel("Residue index")
plt.ylabel("Residue index")
plt.show()

## Distance metrics

In [45]:
u = mda.Universe(pdb_path)
protein = u.select_atoms("protein")

print(f"Number of atoms: {len(protein)}") 
print(f"Center of mass: {protein.center_of_mass()}")

# Center of geometry should be the same as average position of atoms
print(f"Center of geometry: {protein.center_of_geometry()}")
average_position = protein.positions.mean(axis=0)
print(f"Average position of atoms: {average_position}")

# print(f"Positions: {protein.positions}") # all atomic coordinates


ca = u.select_atoms("name CA")
residues = list(zip(ca.resids, ca.resnames))
print(residues)  # print first 10 residues

print(f"ca positions: {ca.positions}")
print(f"ca positions shape: {ca.positions.shape}")

# Some distance metrics

# Calculate Euclidean distance between first and last CA atom
ca_coords = ca.positions
end_to_end = float(np.linalg.norm(ca_coords[0] - ca_coords[-1]))
print(f"End-to-end distance: {end_to_end:.2f} Å")

# Calculate average Euclidean distance between all pairs of CA atoms
dist_array = []
for i in range(len(ca_coords)):
    for j in range(i + 1, len(ca_coords)):
        dist = np.linalg.norm(ca_coords[i] - ca_coords[j])
        dist_array.append(dist)
avg_distance_manual = np.mean(dist_array)
print(f"Average CA-CA distance: {avg_distance_manual:.2f} Å")

# Calculate distance between first and last atom in the entire structure
all_coords = u.atoms.positions # Same as protein.positions
end_to_end_all = float(np.linalg.norm(all_coords[0] - all_coords[-1]))
print(f"End-to-end distance (all atoms): {end_to_end_all:.2f} Å")

# Angle between atoms sliding window of 3 CA atoms
for i in range(len(ca) - 2):
    v1 = ca_coords[i+1] - ca_coords[i]
    v2 = ca_coords[i+2] - ca_coords[i+1]
    cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    angle = np.arccos(cos_angle) * 180 / np.pi
    print(f"Angle between CA {i}, {i+1}, {i+2}: {angle:.2f} degrees")


Number of atoms: 93
Center of mass: [ 1.05875596 -0.54304507  0.11548633]
Center of geometry: [ 1.04864516 -0.53066667  0.02667742]
Average position of atoms: [ 1.0486451  -0.5306665   0.02667758]
[(np.int64(1), 'ILE'), (np.int64(2), 'LYS'), (np.int64(3), 'ARG'), (np.int64(4), 'LEU'), (np.int64(5), 'ASN'), (np.int64(6), 'SER'), (np.int64(7), 'TRP'), (np.int64(8), 'LEU'), (np.int64(9), 'ARG'), (np.int64(10), 'LYS')]
ca positions: [[ 2.33   6.857 -3.612]
 [ 2.022  4.874 -0.373]
 [-0.94   2.983 -1.86 ]
 [ 1.335  0.293 -3.254]
 [ 3.388  0.276 -0.045]
 [ 0.204 -0.264  1.983]
 [-0.556 -3.323 -0.117]
 [ 2.885 -4.697  0.709]
 [ 2.374 -3.884  4.403]
 [-0.272 -6.549  4.845]]
ca positions shape: (10, 3)
End-to-end distance: 16.06 Å
Average CA-CA distance: 6.34 Å
Average CA-CA distance (manual): 7.05 Å
End-to-end distance (all atoms): 17.78 Å
Angle between CA 0, 1, 2: 90.61 degrees
Angle between CA 1, 2, 3: 88.33 degrees
Angle between CA 2, 3, 4: 89.04 degrees
Angle between CA 3, 4, 5: 90.08 degre

In [60]:
BACKBONE_NAMES = {"N", "CA", "C", "O", "OXT"}
u = mda.Universe(pdb_path)
protein = u.select_atoms("protein")

per_res = []
sidechain_point = "com"  # Distance to "cb", "com", or "farthest"
# Iterate residues
for res in protein.residues:
    # CA atom
    ca = res.atoms.select_atoms("name CA")
    if len(ca) != 1:
        continue
    ca_pos = ca.positions[0]

    # Sidechain atom group = residue atoms excluding backbone
    side = res.atoms[[a.name not in BACKBONE_NAMES for a in res.atoms]]

    if len(side) == 0:
        # e.g., glycine may end up empty if you exclude CB-based definitions;
        # or unusual/trimmed residues.
        per_res.append({
            "resid": int(res.resid),
            "resname": str(res.resname),
            "length_A": np.nan,
        })
        continue


    # Beta carbon
    cb = res.atoms.select_atoms("name CB")
    if len(cb) == 1:
        sc_pos = cb.positions[0]
    else: # fallback: com if no CB (glycine, etc.)
        sc_pos = side.center_of_mass()
    v = sc_pos - ca_pos
    length_cb = float(np.linalg.norm(v))

    # Farthest sidechain atom from CA
    d = np.linalg.norm(side.positions - ca_pos, axis=1)
    sc_pos = side.positions[int(np.argmax(d))]
    v = sc_pos - ca_pos
    length_far = float(np.linalg.norm(v))

    # Center of mass of sidechain
    sc_pos = side.center_of_mass()
    v = sc_pos - ca_pos
    length_com = float(np.linalg.norm(v))

    per_res.append({
        "resid": int(res.resid),
        "resname": str(res.resname),
        "length_cb": length_cb,
        "length_far": length_far,
        "length_com": length_com,
    })
print(per_res)

lengths_cb = [r["length_cb"] for r in per_res if not np.isnan(r["length_cb"])]
lengths_far = [r["length_far"] for r in per_res if not np.isnan(r["length_far"])]
lengths_com = [r["length_com"] for r in per_res if not np.isnan(r["length_com"])]

summary = {
    "sidechain_len_mean_cb": float(np.mean(lengths_cb)) if len(lengths_cb) else np.nan,
    "sidechain_len_std_cb": float(np.std(lengths_cb)) if len(lengths_cb) else np.nan,
    "sidechain_len_max_cb": float(np.max(lengths_cb)) if len(lengths_cb) else np.nan,
    "sidechain_len_mean_far": float(np.mean(lengths_far)) if len(lengths_far) else np.nan,
    "sidechain_len_std_far": float(np.std(lengths_far)) if len(lengths_far) else np.nan,
    "sidechain_len_max_far": float(np.max(lengths_far)) if len(lengths_far) else np.nan,
    "sidechain_len_mean_com": float(np.mean(lengths_com)) if len(lengths_com) else np.nan,
    "sidechain_len_std_com": float(np.std(lengths_com)) if len(lengths_com) else np.nan,
    "sidechain_len_max_com": float(np.max(lengths_com)) if len(lengths_com) else np.nan,
    "n_res_measured": int(len(lengths_cb)),
}
print(pd.DataFrame.from_dict(summary, orient="index", columns=["value"]))

[{'resid': 1, 'resname': 'ILE', 'length_cb': 1.5407075881958008, 'length_far': 3.030238151550293, 'length_com': 2.079121720020114}, {'resid': 2, 'resname': 'LYS', 'length_cb': 1.530456781387329, 'length_far': 5.012603759765625, 'length_com': 3.0445677667340068}, {'resid': 3, 'resname': 'ARG', 'length_cb': 1.5286362171173096, 'length_far': 6.609825134277344, 'length_com': 4.031862457795652}, {'resid': 4, 'resname': 'LEU', 'length_cb': 1.533346176147461, 'length_far': 3.125368595123291, 'length_com': 2.3295102200351563}, {'resid': 5, 'resname': 'ASN', 'length_cb': 1.5278223752975464, 'length_far': 3.2541232109069824, 'length_com': 2.519801640777167}, {'resid': 6, 'resname': 'SER', 'length_cb': 1.5221136808395386, 'length_far': 2.407435178756714, 'length_com': 1.9534458134398032}, {'resid': 7, 'resname': 'TRP', 'length_cb': 1.5264455080032349, 'length_far': 6.035944938659668, 'length_com': 3.819541285914774}, {'resid': 8, 'resname': 'LEU', 'length_cb': 1.530342936515808, 'length_far': 3.8

In [48]:
BACKBONE_NAMES = {"N", "CA", "C", "O", "OXT"}

def sidechain_orientation_and_length_features(
    pdb_path: str,
    use_ref: str = "com",          # "com" (protein center-of-mass) is recommended
    sidechain_point: str = "com",  # "cb", "com", or "farthest"
    heavy_only: bool = True,
):
    u = mda.Universe(pdb_path)
    protein = u.select_atoms("protein")
    if len(protein) == 0:
        raise ValueError("No protein atoms found in PDB.")

    prot_com = protein.center_of_mass()

    per_res = []
    lengths = []
    outward_flags = []

    # Iterate residues
    for res in protein.residues:
        # CA atom
        ca = res.atoms.select_atoms("name CA")
        if len(ca) != 1:
            continue
        ca_pos = ca.positions[0]

        # Sidechain atom group = residue atoms excluding backbone
        side = res.atoms[[a.name not in BACKBONE_NAMES for a in res.atoms]]
        if heavy_only and len(side) > 0:
            side = side.select_atoms("not name H*")

        if len(side) == 0:
            # e.g., glycine may end up empty if you exclude CB-based definitions;
            # or unusual/trimmed residues.
            per_res.append({
                "resid": int(res.resid),
                "resname": str(res.resname),
                "length_A": np.nan,
                "outward": np.nan,
            })
            continue

        # Choose sidechain reference point
        if sidechain_point == "cb":
            cb = res.atoms.select_atoms("name CB")
            if len(cb) == 1:
                sc_pos = cb.positions[0]
            else:
                # fallback: COM if no CB (glycine, etc.)
                sc_pos = side.center_of_mass()
        elif sidechain_point == "farthest":
            d = np.linalg.norm(side.positions - ca_pos, axis=1)
            sc_pos = side.positions[int(np.argmax(d))]
        elif sidechain_point == "com":
            sc_pos = side.center_of_mass()
        else:
            raise ValueError("sidechain_point must be one of: 'cb', 'com', 'farthest'.")

        v = sc_pos - ca_pos
        length = float(np.linalg.norm(v))

        # Orientation relative to protein COM
        if use_ref == "com":
            uvec = ca_pos - prot_com  # points from COM to CA (outward radial direction)
            outward = float(np.dot(v, uvec) > 0.0)
        else:
            raise ValueError("use_ref currently supports only 'com'.")

        per_res.append({
            "resid": int(res.resid),
            "resname": str(res.resname),
            "length_A": length,
            "outward": outward,
        })

        lengths.append(length)
        outward_flags.append(outward)

    lengths = np.array([x for x in lengths if np.isfinite(x)], dtype=float)
    outward_flags = np.array(outward_flags, dtype=float)

    summary = {
        "sidechain_len_mean_A": float(np.mean(lengths)) if len(lengths) else np.nan,
        "sidechain_len_std_A": float(np.std(lengths)) if len(lengths) else np.nan,
        "sidechain_len_max_A": float(np.max(lengths)) if len(lengths) else np.nan,
        "sidechain_outward_frac": float(np.mean(outward_flags)) if len(outward_flags) else np.nan,
        "n_res_measured": int(len(lengths)),
    }
    print(summary)
    return per_res, summary
sidechain_orientation_and_length_features(pdb_path)

{'sidechain_len_mean_A': 3.079477622293794, 'sidechain_len_std_A': 0.8946847946132196, 'sidechain_len_max_A': 4.779091681668276, 'sidechain_outward_frac': 0.8, 'n_res_measured': 10}


([{'resid': 1,
   'resname': 'ILE',
   'length_A': 2.079121720020114,
   'outward': 1.0},
  {'resid': 2,
   'resname': 'LYS',
   'length_A': 3.0445677667340068,
   'outward': 1.0},
  {'resid': 3,
   'resname': 'ARG',
   'length_A': 4.031862457795652,
   'outward': 1.0},
  {'resid': 4,
   'resname': 'LEU',
   'length_A': 2.3295102200351563,
   'outward': 1.0},
  {'resid': 5,
   'resname': 'ASN',
   'length_A': 2.519801640777167,
   'outward': 1.0},
  {'resid': 6,
   'resname': 'SER',
   'length_A': 1.9534458134398032,
   'outward': 1.0},
  {'resid': 7,
   'resname': 'TRP',
   'length_A': 3.819541285914774,
   'outward': 1.0},
  {'resid': 8,
   'resname': 'LEU',
   'length_A': 2.6126993292883647,
   'outward': 1.0},
  {'resid': 9,
   'resname': 'ARG',
   'length_A': 4.779091681668276,
   'outward': 0.0},
  {'resid': 10,
   'resname': 'LYS',
   'length_A': 3.6251343072646263,
   'outward': 0.0}],
 {'sidechain_len_mean_A': 3.079477622293794,
  'sidechain_len_std_A': 0.8946847946132196,
  '

## HELANAL

In [42]:
u = mda.Universe(pdb_path)

# Assume the entire protein is a helix
# Use name CA because HELANAL operates on CA atoms.
helanal = hel.HELANAL(u, select="name CA")
helanal.run()
print(helanal.results.summary)



{'local_twists': {'mean': array([120.11676788,  95.83618164, 101.38298798, 102.87566376,
       102.15701294,  99.02890015, 107.71524811]), 'sample_sd': array([nan, nan, nan, nan, nan, nan, nan]), 'abs_dev': array([0., 0., 0., 0., 0., 0., 0.])}, 'local_bends': {'mean': array([27.53843498,  7.29691696,  4.11865568, 15.64727783]), 'sample_sd': array([nan, nan, nan, nan]), 'abs_dev': array([0., 0., 0., 0.])}, 'local_heights': {'mean': array([2.2160418 , 1.30698824, 1.56792128, 1.58704627, 1.55290604,
       1.40947855, 1.92259049]), 'sample_sd': array([nan, nan, nan, nan, nan, nan, nan]), 'abs_dev': array([0., 0., 0., 0., 0., 0., 0.])}, 'local_nres_per_turn': {'mean': array([2.99708366, 3.75640988, 3.55089164, 3.49936986, 3.52398705,
       3.63530231, 3.3421452 ]), 'sample_sd': array([nan, nan, nan, nan, nan, nan, nan]), 'abs_dev': array([0., 0., 0., 0., 0., 0., 0.])}, 'local_origins': {'mean': array([[ 1.1484201 ,  4.90428257, -1.92859077],
       [ 1.4433198 ,  2.61938095, -1.81767642]